In [ ]:
import os
import glob
import pandas as pd
import re
import numpy as np

def procesar_pipeline_proyecto_02(ruta_busqueda):
    """
    Script para el Pipeline del Proyecto 02.

    Objetivo:
    - Leer 12 archivos Excel (ER_MM-YYYY_SUPERFICIE.XLS).
    - Extraer m2 de la hoja 'SUP_PINTADA'.
    - Extraer ingresos (MALBEC + NETO) por cliente de 'ESTADO_DE_RESULTADO'.
    - Extraer cv_mod (fila MOD, MALBEC + NETO) de 'ESTADO_DE_RESULTADO'.
    - Consolidar todo en un DataFrame por artículo-mes.
    """

    # Listar los archivos XLS que coincidan con el patrón
    archivos = glob.glob(os.path.join(ruta_busqueda, "ER_*-*_SUPERFICIE.XLS*"))

    if not archivos:
        print("No se encontraron archivos en la ruta.")
        return pd.DataFrame()

    print(f"Se encontraron {len(archivos)} archivos. Procesando...")
    dfs_meses = []

    for ruta in archivos:
        nombre_archivo = os.path.basename(ruta)
        # Capturar el mes/año del nombre del archivo para la columna 'mes'
        match_fecha = re.search(r'(\d{2}-\d{4})', nombre_archivo)
        mes_valor = match_fecha.group(1) if match_fecha else "Desconocido"

        try:
            # --- 1. PROCESAR HOJA: SUP_PINTADA ---
            df_sup_raw = pd.read_excel(ruta, sheet_name='SUP_PINTADA')
            df_sup_raw.columns = df_sup_raw.columns.astype(str).str.strip()

            # Selección y renombre de columnas según Criterios de Aceptación
            columnas_sup = {
                'ARTICULO': 'articulo',
                'DESCRIPCIÓN': 'descripcion',
                'CLIENTE': 'cod_cliente',
                'NOMBRE': 'cliente',
                'TOTAL SUP.': 'm2'
            }
            df_sup = df_sup_raw[list(columnas_sup.keys())].copy()
            df_sup.rename(columns=columnas_sup, inplace=True)

            # Limpieza de tipos de datos
            df_sup['cod_cliente'] = df_sup['cod_cliente'].astype(str).str.replace('.0', '', regex=False)
            df_sup['articulo'] = pd.to_numeric(df_sup['articulo'], errors='coerce').fillna(0).astype(int).astype(str)

            # --- 2. PROCESAR HOJA: ESTADO_DE_RESULTADO ---
            # Leemos sin encabezado para buscar filas específicas
            df_er_raw = pd.read_excel(ruta, sheet_name='ESTADO_DE_RESULTADO', header=None)

            # A. Detectar fila de encabezados buscando 'MALBEC'
            idx_header = 0
            for i, fila in df_er_raw.iterrows():
                if fila.astype(str).str.contains('MALBEC', case=False).any():
                    idx_header = i
                    break

            # Cargar datos financieros con el encabezado correcto
            df_finan_data = pd.read_excel(ruta, sheet_name='ESTADO_DE_RESULTADO', header=idx_header)
            df_finan_data.columns = df_finan_data.columns.astype(str).str.strip()

            # B. Calcular Ingresos por Cliente (MALBEC + NETO)
            df_finan_data['MALBEC'] = pd.to_numeric(df_finan_data['MALBEC'], errors='coerce').fillna(0)
            df_finan_data['NETO'] = pd.to_numeric(df_finan_data['NETO'], errors='coerce').fillna(0)
            df_finan_data['ingresos_calculados'] = df_finan_data['MALBEC'] + df_finan_data['NETO']

            # Limpiar ID de cliente para el cruce
            df_finan_data['CLIENTE'] = df_finan_data['CLIENTE'].astype(str).str.replace('.0', '', regex=False)
            mapa_ingresos = df_finan_data.groupby('CLIENTE')['ingresos_calculados'].sum().to_dict()

            # C. Extraer cv_mod (Fila 'MOD', suma de sus columnas MALBEC y NETO)
            # Buscamos 'MOD' en la primera columna del archivo crudo
            fila_mod = df_er_raw[df_er_raw[0].astype(str).str.strip().str.upper() == 'MOD']
            if not fila_mod.empty:
                val_malbec_mod = pd.to_numeric(fila_mod.iloc[0, 2], errors='coerce')
                val_neto_mod = pd.to_numeric(fila_mod.iloc[0, 3], errors='coerce')
                cv_mod_total_mes = np.nansum([val_malbec_mod, val_neto_mod])
            else:
                cv_mod_total_mes = 0

            # --- 3. CONSOLIDACIÓN FINAL DEL MES ---
            df_sup['mes'] = mes_valor
            df_sup['ingresos'] = df_sup['cod_cliente'].map(mapa_ingresos).fillna(0)
            df_sup['cv_mod'] = cv_mod_total_mes

            dfs_meses.append(df_sup)
            print(f"  [OK] Procesado: {nombre_archivo}")

        except Exception as e:
            print(f"  [ERROR] Falló procesamiento de {nombre_archivo}: {e}")

    # Concatenar todos los meses
    if dfs_meses:
        df_er = pd.concat(dfs_meses, ignore_index=True)

        # Ordenar columnas finales
        cols = ['mes', 'articulo', 'descripcion', 'cod_cliente', 'cliente', 'm2', 'ingresos', 'cv_mod']
        df_er = df_er[cols]

        # Guardar resultado para el pipeline con formato regional
        df_er.to_csv("pipeline_proyecto_02_consolidado.csv", index=False, sep=';', decimal=',', float_format='%.2f')
        print(f"\nProceso terminado. Archivo 'pipeline_proyecto_02_consolidado.csv' generado con {len(df_er)} registros.")
        return df_er
    else:
        return pd.DataFrame()

# Ejecución del pipeline
df_er = procesar_pipeline_proyecto_02("/content/")
display(df_er.head())

Se encontraron 12 archivos. Procesando...
  [OK] Procesado: ER_11-2025_SUPERFICIE.XLS
  [OK] Procesado: ER_07-2025_SUPERFICIE.XLS
  [OK] Procesado: ER_10-2025_SUPERFICIE.XLS
  [OK] Procesado: ER_06-2025_SUPERFICIE.XLS
  [OK] Procesado: ER_08-2025_SUPERFICIE.XLS
  [OK] Procesado: ER_09-2025_SUPERFICIE.XLS
  [OK] Procesado: ER_02-2025_SUPERFICIE.XLS
  [OK] Procesado: ER_05-2025_SUPERFICIE.XLS
  [OK] Procesado: ER_01-2025_SUPERFICIE.XLS
  [OK] Procesado: ER_04-2025_SUPERFICIE.XLS
  [OK] Procesado: ER_03-2025_SUPERFICIE.XLS
  [OK] Procesado: ER_12-2025_SUPERFICIE.XLS

Proceso terminado. Archivo 'pipeline_proyecto_02_consolidado.csv' generado con 807 registros.


,mes,articulo,descripcion,cod_cliente,cliente,m2,ingresos,cv_mod
0,11-2025,306,SOPORTE GUARDABARRO PARA TANQUE Y TOLVA (GRAFITO),10016,DANES SRL,22.00,10130646.49,4505180.92
1,11-2025,455,SOPORTE VIRAJE IZQUIERDO (GRAFITO),10016,DANES SRL,9.72,10130646.49,4505180.92
2,11-2025,456,SOPORTE VIRAJE DERECHO (GRAFITO),10016,DANES SRL,9.72,10130646.49,4505180.92
3,11-2025,163,RIENDA SOPORTE TRAVESAÑO PORTA AUX (ESTRELLITA),10016,DANES SRL,63.00,10130646.49,4505180.92
4,11-2025,758,0132 -REFUERZO W S05,10049,AFG INGENIERIA SRL,30.40,10885437.00,4505180.92
